In [0]:
df_customer_info = spark.read.table("db_project.silver.crm_cust_info")
df_customer_info_gnd = spark.read.table("db_project.silver.erp_cust_az12")
df_customer_info_loc = spark.read.table("db_project.silver.erp_loc_a101")
df_customer_info.limit(1).display()
df_customer_info_gnd.limit(1).display()
df_customer_info_loc.limit(1).display()

In [0]:
outcome = spark.sql("""
    SELECT 
        ci.cst_id, 
        ci.cst_key, 
        ci.cst_firstname, 
        ci.cst_lastname, 
        ci.cst_marital_status, 
        ci.cst_gndr,
        ci.cst_create_date, 
        cg.CID, 
        cg.gen,
        cg.bdate, 
        cl.CID, 
        cl.CNTRY
    FROM db_project.silver.crm_cust_info ci
    LEFT JOIN db_project.silver.erp_cust_az12 cg
    ON ci.cst_key = cg.CID
    LEFT JOIN db_project.silver.erp_loc_a101 cl
    ON ci.cst_key = cl.CID
""")
outcome.display()

In [0]:
outcome.select("cst_gndr", "gen").distinct().display()

In [0]:
outcome = spark.sql("""
    SELECT 
        ci.cst_id,
        CASE 
            WHEN ci.cst_gndr == "n/a" THEN cg.gen
            WHEN cg.gen == "n/a" THEN ci.cst_gndr
            ELSE ci.cst_gndr
        END AS gender,
        ci.cst_gndr,
        cg.gen
    FROM db_project.silver.crm_cust_info ci
    LEFT JOIN db_project.silver.erp_cust_az12 cg
    ON ci.cst_key = cg.CID
    LEFT JOIN db_project.silver.erp_loc_a101 cl
    ON ci.cst_key = cl.CID
""")
outcome.select("gender").distinct().display()

In [0]:
outcome = spark.sql("""
    SELECT 
        ROW_NUMBER() OVER(ORDER BY ci.cst_create_date) AS Customer_key,
        ci.cst_id AS Customer_id, 
        ci.cst_key AS Customer_number, 
        ci.cst_firstname AS First_name, 
        ci.cst_lastname AS Last_name, 
        CASE 
            WHEN ci.cst_gndr == "n/a" THEN cg.gen
            WHEN cg.gen == "n/a" THEN ci.cst_gndr
            ELSE ci.cst_gndr
        END AS Gender,
        cg.bdate AS Birth_date,
        cl.CNTRY AS Country,
        ci.cst_marital_status AS Marital_status, 
        ci.cst_create_date AS Date_created
    FROM db_project.silver.crm_cust_info ci
    LEFT JOIN db_project.silver.erp_cust_az12 cg
    ON ci.cst_key = cg.CID
    LEFT JOIN db_project.silver.erp_loc_a101 cl
    ON ci.cst_key = cl.CID
""")
outcome.display()

In [0]:
outcome.write.mode("overwrite").format("delta").saveAsTable("db_project.gold.dim_customer_info")